# Speech Emotion Recognition with Qwen2-Audio
Lädt das EMoDB-Dataset direkt von Kaggle herunter. 
Benötigt einen Kaggle-Account und API-Key: https://www.kaggle.com/settings → API → Create New Token

In [1]:
import sys
!{sys.executable} -m pip install --user "numpy<2"


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, torch
print("python:", sys.executable)
print("torch :", torch.__version__, "@", torch.__file__)
!{sys.executable} -m pip show torch  2>&1 | head -5
!{sys.executable} -m pip show torchvision 2>&1 | head -5
!ls ~/.local/lib/python3.12/site-packages/ | grep -iE "^torch"

python: /sw/ubuntu22/generic/python/3.12.9_dir/bin/python3
torch : 2.11.0+cu130 @ /data/homes/jteklege/.local/lib/python3.12/site-packages/torch/__init__.py
Name: torch
Version: 2.11.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Name: torchvision
Version: 0.26.0+cu130
Summary: image and video datasets and models for torch deep learning
Home-page: https://github.com/pytorch/vision
Author: PyTorch Core Team
torch
torch-2.11.0.dist-info
torchgen
torchvision
torchvision-0.26.0+cu130.dist-info
torchvision.libs


In [3]:
!pip install -q kaggle transformers accelerate librosa


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import os

# Kaggle-Zugangsdaten hier eintragen
os.environ["KAGGLE_USERNAME"] = "DEIN_USERNAME"
os.environ["KAGGLE_KEY"]      = "DEIN_API_KEY"

# Kaggle-Dataset-Slug (den Teil der URL nach kaggle.com/datasets/)
DATASET_SLUG = "piyushagni5/berlin-database-of-emotional-speech-emodb"
DOWNLOAD_DIR = "/tmp/emodb"

# Prompt-Modus: 'normal' oder 'prosodic'
PROMPT_MODE = "prosodic"

In [6]:
# Dataset von Kaggle herunterladen und entpacken
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --unzip

from pathlib import Path
all_files = sorted(Path(DOWNLOAD_DIR).rglob("*.wav"))
print(f"Gefundene WAV-Dateien: {len(all_files)}")
for f in all_files[:5]:
    print(f"  {f}")

/sw/ubuntu22/generic/python/3.11.8_dir/bin/python3.11: error while loading shared libraries: libpython3.11.so.1.0: cannot open shared object file: No such file or directory
Gefundene WAV-Dateien: 0


In [10]:
! python3 -m pip install --user --upgrade torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import sys
!{sys.executable} -m pip install --user --force-reinstall "transformers==4.48.3"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached transformers-4.48.3-py3-none-any.whl.metadata (44 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached packaging-26.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.4.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-202

In [12]:
import json
import torch
import librosa
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

PROMPTS = {
    "normal": (
        "Listen to this audio clip and classify the emotion of the speaker. "
        "Choose from: anger, anxiety, boredom, disgust, fear, happiness, neutral, sadness. "
        "Reply with just the emotion label."
    ),
    "prosodic": (
        "Listen to this audio clip and classify the emotion of the speaker "
        "based only on prosodic and acoustic features such as pitch, "
        "speech rate, rhythm and loudness. "
        "Ignore the spoken words and linguistic content entirely. "
        "Choose from: anger, anxiety, boredom, disgust, fear, happiness, neutral, sadness. "
        "Reply with just the emotion label."
    ),
}

PROMPT = PROMPTS[PROMPT_MODE]
output_file = f"emotion_results_qwen_{PROMPT_MODE}.json"

# Resume
results = {}
if os.path.exists(output_file):
    with open(output_file) as f:
        results = json.load(f)
    print(f"{len(results)} Dateien bereits verarbeitet")

# Modell laden
print("Lade Modell...")
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
SR = processor.feature_extractor.sampling_rate
print(f"Modell geladen auf: {next(model.parameters()).device}")

Lade Modell...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Modell geladen auf: cpu


In [ ]:
total_files = len(all_files)

for i, audio_file in enumerate(all_files, 1):
    if audio_file.name in results:
        print(f"({i}/{total_files}) Übersprungen: {audio_file.name}")
        continue

    print(f"({i}/{total_files}) {audio_file.name}", end=" ... ")
    try:
        audio, _ = librosa.load(str(audio_file), sr=SR)

        conversation = [{"role": "user", "content": [
            {"type": "audio", "audio_url": str(audio_file)},
            {"type": "text",  "text": PROMPT},
        ]}]

        text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, audios=[audio], sampling_rate=SR, return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        generate_ids = model.generate(**inputs, max_new_tokens=10)
        generate_ids = generate_ids[:, inputs["input_ids"].size(1):]
        emotion = processor.batch_decode(generate_ids, skip_special_tokens=True)[0].strip()

        results[audio_file.name] = emotion
        print(emotion)

        with open(output_file, "w") as f:
            json.dump(results, f, indent=2)

    except Exception as e:
        print(f"FEHLER: {e}")

print(f"\nFertig. {len(results)}/{total_files} Dateien verarbeitet.")

(1/535) 03a01Fa.wav ... 

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


Happy
Neutral 03a01Nc.wav ... 
Angry5) 03a01Wa.wav ... 
Happy5) 03a02Fc.wav ... 
Neutral 03a02Nc.wav ... 
Sad535) 03a02Ta.wav ... 
Angry5) 03a02Wb.wav ... 
Angry5) 03a02Wc.wav ... 
Fearful 03a04Ad.wav ... 
Neutral) 03a04Fd.wav ... 
neutral) 03a04Lc.wav ... 
Neutral) 03a04Nc.wav ... 
neutral) 03a04Ta.wav ... 
Angry35) 03a04Wc.wav ... 
Fearful) 03a05Aa.wav ... 
Angry35) 03a05Fc.wav ... 
Neutral) 03a05Nd.wav ... 
Neutral) 03a05Tc.wav ... 
Angry35) 03a05Wa.wav ... 
Angry35) 03a05Wb.wav ... 
Fearful) 03a07Fa.wav ... 
Fearful) 03a07Fb.wav ... 
neutral) 03a07La.wav ... 
Neutral) 03a07Nc.wav ... 
Angry35) 03a07Wc.wav ... 
Fearful) 03b01Fa.wav ... 
Neutral) 03b01Lb.wav ... 
Neutral) 03b01Nb.wav ... 
Sad/535) 03b01Td.wav ... 
Angry35) 03b01Wa.wav ... 
Angry35) 03b01Wc.wav ... 
Fearful) 03b02Aa.wav ... 
Fearful) 03b02La.wav ... 
Fearful) 03b02Na.wav ... 
Sadness) 03b02Tb.wav ... 
Angry35) 03b02Wb.wav ... 
Neutral) 03b03Nb.wav ... 
Sad/535) 03b03Tc.wav ... 
Angry35) 03b03Wc.wav ... 
Neutral) 03b09

In [ ]:
  !pip install --user --upgrade torchvision --index-url https://download.pytorch.org/whl/cu130

In [ ]:
# Ergebnisse anzeigen und als CSV speichern
import pandas as pd

df = pd.DataFrame(list(results.items()), columns=["file", "emotion"])
print(df["emotion"].value_counts().to_string())

csv_file = output_file.replace(".json", ".csv")
df.to_csv(csv_file, index=False)
print(f"\nCSV gespeichert: {csv_file}")